# 生产监控教程

> **前置知识**: Python基础、模型部署概念（参见 02_ModelRegistry_tutorial）
>
> **学习目标**: 掌握生产环境中模型监控的核心方法

---

## 为什么需要生产监控？

```
模型上线后的问题:
┌─────────────────────────────────────────────────────────────┐
│  "模型上线后效果怎么样？"     → 没有数据                    │
│  "响应时间是否正常？"         → 不知道                      │
│  "什么时候需要重训练？"       → 没有告警                    │
└─────────────────────────────────────────────────────────────┘

生产监控解决方案:
┌─────────────────────────────────────────────────────────────┐
│  指标收集 → 延迟、吞吐量、错误率、准确率                    │
│  漂移检测 → 发现数据分布变化                                │
│  告警管理 → 异常时自动通知                                  │
│  可视化   → 仪表盘展示趋势                                  │
└─────────────────────────────────────────────────────────────┘
```

## 监控指标分类

```
┌─────────────────────────────────────────────────────────────┐
│  系统指标              模型指标              业务指标        │
│  (基础设施)           (模型表现)            (业务价值)       │
│  ──────────           ──────────            ──────────      │
│  CPU使用率            预测延迟              转化率          │
│  内存使用率           吞吐量(QPS)           点击率          │
│  GPU利用率            错误率                收入            │
│  网络带宽             准确率/漂移           用户满意度      │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **告警级别** - 理解告警严重程度
2. **漂移检测** - KS检验、PSI等方法
3. **指标监控** - 延迟、错误率统计
4. **告警管理** - 规则配置和触发
5. **模型监控** - 综合监控方案
6. **Prometheus** - 企业级集成

In [ ]:
# ============================================================
# 环境准备
# ============================================================
# 添加源码路径，使得可以导入 src 目录下的模块
import sys
sys.path.insert(0, '../src')

# 标准库
import time
import random
import statistics

# 导入监控模块
from monitoring import (
    MetricType,          # 指标类型枚举：COUNTER, GAUGE, HISTOGRAM
    AlertSeverity,       # 告警级别枚举：INFO, WARNING, ERROR, CRITICAL
    Alert,               # 告警数据类
    DriftDetector,       # 漂移检测器：KS检验、PSI等
    MetricsMonitor,      # 指标监控器：延迟、错误率统计
    ModelMonitor,        # 模型监控器：综合监控
    create_monitor,      # 工厂函数：创建监控器
    PROMETHEUS_AVAILABLE,  # Prometheus 是否可用
)

# 检查可选依赖
print("=" * 50)
print("环境检查")
print("=" * 50)
print(f"Prometheus 可用: {PROMETHEUS_AVAILABLE}")
print(f"Python 版本: {sys.version.split()[0]}")

## 1. 告警级别和指标类型

**告警级别**: 根据问题严重程度分级，便于优先处理

```
告警级别优先级:
┌─────────────────────────────────────────────────────────────┐
│  CRITICAL → 立即处理（服务不可用）                          │
│  ERROR    → 尽快处理（功能受损）                            │
│  WARNING  → 关注（可能恶化）                                │
│  INFO     → 记录（正常事件）                                │
└─────────────────────────────────────────────────────────────┘
```

**指标类型**: 不同类型的指标有不同的聚合方式
- **COUNTER**: 只增不减（如请求总数）
- **GAUGE**: 可增可减（如当前内存使用）
- **HISTOGRAM**: 分布统计（如延迟分布）

In [ ]:
# ============================================================
# 查看告警级别和指标类型
# ============================================================
# AlertSeverity 定义告警的严重程度
# MetricType 定义指标的类型（影响聚合方式）

print("=" * 50)
print("告警级别 (AlertSeverity)")
print("=" * 50)
for severity in AlertSeverity:
    # 根据级别添加说明
    desc = {
        "INFO": "记录事件，无需处理",
        "WARNING": "需要关注，可能恶化",
        "ERROR": "功能受损，尽快处理",
        "CRITICAL": "服务不可用，立即处理"
    }.get(severity.name, "")
    print(f"  {severity.name:10} → {desc}")

print("\n" + "=" * 50)
print("指标类型 (MetricType)")
print("=" * 50)
for metric_type in MetricType:
    desc = {
        "COUNTER": "只增不减（如请求总数）",
        "GAUGE": "可增可减（如内存使用）",
        "HISTOGRAM": "分布统计（如延迟分布）"
    }.get(metric_type.name, "")
    print(f"  {metric_type.name:10} → {desc}")

## 2. 数据漂移检测 (DriftDetector)

**核心概念**: 数据漂移是指生产数据分布与训练数据分布发生变化

```
漂移检测方法:
┌─────────────────────────────────────────────────────────────┐
│  KS检验 (Kolmogorov-Smirnov)                                │
│  - 比较两个分布的累积分布函数(CDF)的最大差异                │
│  - 返回 p值，p < 0.05 表示分布显著不同                      │
│                                                             │
│  PSI (Population Stability Index)                           │
│  - 比较两个分布在各区间的占比差异                           │
│  - PSI < 0.1: 无变化, 0.1-0.2: 轻微, > 0.2: 显著           │
│                                                             │
│  均值/方差检测                                               │
│  - 简单比较均值或方差的相对变化                             │
│  - 适合快速检测，但可能漏检分布形状变化                     │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 创建漂移检测器
# ============================================================
# DriftDetector 需要参考数据（训练时的数据分布）作为基准
# 后续会用这个基准来检测新数据是否发生漂移

# 创建参考数据（模拟训练时的数据分布）
# 正态分布：均值50，标准差10
reference_data = [random.gauss(50, 10) for _ in range(1000)]

# 创建漂移检测器
detector = DriftDetector(reference_data=reference_data)

print("=" * 50)
print("漂移检测器创建成功")
print("=" * 50)
print(f"参考数据样本数: {len(reference_data)}")
print(f"参考数据均值:   {statistics.mean(reference_data):.2f}")
print(f"参考数据标准差: {statistics.stdev(reference_data):.2f}")
print(f"\n支持的检测方法: ks, psi, mean, std")

In [ ]:
# ============================================================
# KS检验 - 无漂移情况
# ============================================================
# 创建与参考数据相似的新数据（相同分布）
# 预期结果：不检测到漂移

similar_data = [random.gauss(50, 10) for _ in range(500)]  # 相同分布

# 使用 KS 检验检测漂移
# method="ks": Kolmogorov-Smirnov 检验
result = detector.detect_drift(similar_data, method="ks")

print("=" * 50)
print("KS检验 - 相似数据（无漂移）")
print("=" * 50)
print(f"当前数据均值: {statistics.mean(similar_data):.2f}")
print(f"当前数据标准差: {statistics.stdev(similar_data):.2f}")
print("-" * 50)
print(f"漂移检测: {'是 ⚠️' if result['drift_detected'] else '否 ✓'}")
print(f"KS统计量: {result.get('statistic', 'N/A'):.4f}")
print(f"p值: {result.get('p_value', 'N/A'):.4f}")
print("\n说明: p值 > 0.05 表示两个分布没有显著差异")

In [ ]:
# ============================================================
# KS检验 - 有漂移情况
# ============================================================
# 创建与参考数据不同的新数据（均值偏移）
# 预期结果：检测到漂移

drifted_data = [random.gauss(70, 10) for _ in range(500)]  # 均值从50变为70

# 使用 KS 检验检测漂移
result = detector.detect_drift(drifted_data, method="ks")

print("=" * 50)
print("KS检验 - 漂移数据（均值偏移）")
print("=" * 50)
print(f"参考数据均值: ~50")
print(f"当前数据均值: {statistics.mean(drifted_data):.2f} (偏移了+20)")
print("-" * 50)
print(f"漂移检测: {'是 ⚠️' if result['drift_detected'] else '否 ✓'}")
print(f"KS统计量: {result.get('statistic', 'N/A'):.4f}")
print(f"p值: {result.get('p_value', 'N/A'):.6f}")
print("\n说明: p值 < 0.05 表示两个分布有显著差异")

In [ ]:
# ============================================================
# PSI检验 - 对比两种数据
# ============================================================
# PSI (Population Stability Index) 是另一种常用的漂移检测方法
# 它将数据分成多个区间，比较各区间的占比变化

print("=" * 50)
print("PSI检验对比")
print("=" * 50)

# 相似数据的 PSI
result1 = detector.detect_drift(similar_data, method="psi", threshold=0.2)
print(f"相似数据 PSI: {result1.get('psi', 0):.4f}")
print(f"  漂移检测: {'是 ⚠️' if result1['drift_detected'] else '否 ✓'}")

# 漂移数据的 PSI
result2 = detector.detect_drift(drifted_data, method="psi", threshold=0.2)
print(f"\n漂移数据 PSI: {result2.get('psi', 0):.4f}")
print(f"  漂移检测: {'是 ⚠️' if result2['drift_detected'] else '否 ✓'}")

print("\n" + "-" * 50)
print("PSI 解释标准:")
print("  < 0.1  : 无显著变化 ✓")
print("  0.1-0.2: 轻微变化，需关注")
print("  > 0.2  : 显著变化，需要行动 ⚠️")

In [ ]:
# ============================================================
# 均值检测 - 简单快速的漂移检测
# ============================================================
# 均值检测是最简单的漂移检测方法
# 适合快速检测，但可能漏检分布形状变化（如方差变化）

result = detector.detect_drift(drifted_data, method="mean", threshold=0.1)

print("=" * 50)
print("均值检测 - 漂移数据")
print("=" * 50)
print(f"参考数据均值: {result['reference_mean']:.2f}")
print(f"当前数据均值: {result['current_mean']:.2f}")
print(f"相对差异:     {result['relative_diff']:.2%}")
print(f"阈值:         {0.1:.2%}")
print("-" * 50)
print(f"漂移检测: {'是 ⚠️' if result['drift_detected'] else '否 ✓'}")
print("\n说明: 相对差异 > 阈值 时判定为漂移")

## 3. MetricsMonitor 指标监控器

**核心功能**: 收集和统计服务运行时的关键指标

```
MetricsMonitor 功能:
┌─────────────────────────────────────────────────────────────┐
│  record_latency()  → 记录请求延迟                           │
│  record_error()    → 记录错误                               │
│  get_stats()       → 获取统计信息                           │
│  add_alert_rule()  → 添加告警规则                           │
│  check_alerts()    → 检查告警                               │
└─────────────────────────────────────────────────────────────┘

关键指标:
┌─────────────────────────────────────────────────────────────┐
│  延迟百分位数 (Percentile):                                  │
│  - P50 (中位数): 50%的请求在此时间内完成                    │
│  - P90: 90%的请求在此时间内完成                             │
│  - P99: 99%的请求在此时间内完成 ← 最常用的SLA指标          │
│                                                             │
│  为什么用P99而不是平均值？                                   │
│  → 平均值会被极端值拉高，无法反映大多数用户体验             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 创建指标监控器
# ============================================================
# MetricsMonitor 使用滑动窗口存储最近的指标数据
# window_size 决定保留多少条记录（避免内存无限增长）

monitor = MetricsMonitor(window_size=10000)  # 保留最近10000条记录

print("=" * 50)
print("MetricsMonitor 创建成功")
print("=" * 50)
print("核心功能:")
print("  - record_latency()  记录请求延迟")
print("  - record_error()    记录错误")
print("  - get_stats()       获取统计信息")
print("  - add_alert_rule()  添加告警规则")
print("  - check_alerts()    检查告警")

In [ ]:
# ============================================================
# 模拟请求并记录指标
# ============================================================
# 模拟生产环境中的请求，记录延迟和错误

print("=" * 50)
print("模拟 1000 个请求...")
print("=" * 50)

for i in range(1000):
    # 模拟延迟（指数分布，平均50ms）
    # 指数分布常用于模拟请求延迟，因为大多数请求快，少数请求慢
    latency = random.expovariate(1/50)  # 平均 50ms
    
    # 模拟 5% 错误率
    if random.random() < 0.05:
        # 记录错误
        monitor.record_error("Timeout error")
    else:
        # 记录成功请求的延迟
        monitor.record_latency(latency)

print("请求模拟完成!")
print(f"  成功请求: ~950")
print(f"  失败请求: ~50 (5%错误率)")

In [ ]:
# ============================================================
# 获取统计信息
# ============================================================
# get_stats() 返回所有收集的指标统计
# 包括请求数、错误率、延迟百分位数等

stats = monitor.get_stats()

print("=" * 50)
print("服务统计信息")
print("=" * 50)
print(f"总请求数:   {stats['total_requests']}")
print(f"成功请求:   {stats['successful_requests']}")
print(f"失败请求:   {stats['failed_requests']}")
print(f"错误率:     {stats['error_rate']:.2%}")

print("\n" + "-" * 50)
print("延迟统计 (ms):")
print("-" * 50)
print(f"  平均值: {stats.get('latency_avg', 0):.2f}")
print(f"  P50:    {stats.get('latency_p50', 0):.2f}  ← 50%请求在此时间内完成")
print(f"  P90:    {stats.get('latency_p90', 0):.2f}  ← 90%请求在此时间内完成")
print(f"  P99:    {stats.get('latency_p99', 0):.2f}  ← 99%请求在此时间内完成")

print("\n提示: P99 是最常用的 SLA 指标，反映长尾延迟")

## 4. 告警规则管理

**核心功能**: 当指标超过阈值时自动触发告警

```
告警规则配置:
┌─────────────────────────────────────────────────────────────┐
│  name       → 规则名称（唯一标识）                          │
│  metric     → 监控的指标名称                                │
│  condition  → 条件：gt(大于), lt(小于), eq(等于)           │
│  threshold  → 阈值                                          │
│  severity   → 告警级别                                      │
│  message    → 告警消息                                      │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 添加告警规则
# ============================================================
# add_alert_rule() 定义当指标超过阈值时触发告警
# 可以配置多个规则，监控不同的指标

# 规则1: P99延迟告警
monitor.add_alert_rule(
    name="high_latency",           # 规则名称（唯一标识）
    metric="latency_p99",          # 监控的指标
    condition="gt",                # 条件：gt=大于, lt=小于, eq=等于
    threshold=100.0,               # 阈值：100ms
    severity=AlertSeverity.WARNING,  # 告警级别
    message="P99 延迟超过 100ms，可能影响用户体验"
)

# 规则2: 错误率告警
monitor.add_alert_rule(
    name="high_error_rate",
    metric="error_rate",
    condition="gt",
    threshold=0.01,                # 阈值：1%
    severity=AlertSeverity.ERROR,  # 更高的告警级别
    message="错误率超过 1%，需要立即排查"
)

print("=" * 50)
print("已添加告警规则")
print("=" * 50)
print("规则1: high_latency")
print("  条件: P99延迟 > 100ms")
print("  级别: WARNING")
print("\n规则2: high_error_rate")
print("  条件: 错误率 > 1%")
print("  级别: ERROR")

In [ ]:
# ============================================================
# 检查告警
# ============================================================
# check_alerts() 检查所有规则，返回触发的告警列表
# 在生产环境中，通常定期调用此方法（如每分钟）

alerts = monitor.check_alerts()

print("=" * 50)
print(f"告警检查结果: 触发 {len(alerts)} 个告警")
print("=" * 50)

for alert in alerts:
    # 根据级别显示不同的标记
    level_mark = {
        "INFO": "ℹ️",
        "WARNING": "⚠️",
        "ERROR": "❌",
        "CRITICAL": "🔥"
    }.get(alert.severity.value.upper(), "")
    
    print(f"\n{level_mark} [{alert.severity.value.upper()}] {alert.name}")
    print(f"   消息: {alert.message}")
    print(f"   指标: {alert.metric_name} = {alert.metric_value:.4f}")
    print(f"   阈值: {alert.threshold}")

In [ ]:
# ============================================================
# 告警管理：解决和查询
# ============================================================
# 告警触发后需要处理，处理完成后标记为已解决
# 可以查询未解决和已解决的告警

if alerts:
    # 解决第一个告警
    monitor.resolve_alert(alerts[0].name)
    print(f"已解决告警: {alerts[0].name}")

# 查看告警状态
unresolved = monitor.get_alerts(resolved=False)
resolved = monitor.get_alerts(resolved=True)

print("\n" + "=" * 50)
print("告警状态统计")
print("=" * 50)
print(f"未解决告警: {len(unresolved)} 个")
print(f"已解决告警: {len(resolved)} 个")

if unresolved:
    print("\n未解决告警列表:")
    for alert in unresolved:
        print(f"  - {alert.name}: {alert.message}")

## 5. ModelMonitor 模型监控器

**核心功能**: 综合监控模型的性能，包括延迟、准确率、漂移等

```
ModelMonitor = MetricsMonitor + DriftDetector + 模型元数据
┌─────────────────────────────────────────────────────────────┐
│  record_inference()  → 记录每次推理                         │
│  get_stats()         → 获取综合统计                         │
│  check_drift()       → 检查数据漂移                         │
│  get_prediction_distribution() → 获取预测分布               │
│  export_metrics()    → 导出指标（JSON/Prometheus）          │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 创建模型监控器
# ============================================================
# ModelMonitor 是综合监控器，结合了指标监控和漂移检测
# 需要提供参考数据用于漂移检测

# 创建参考数据（模拟训练时的预测分布）
reference = [random.gauss(0.5, 0.1) for _ in range(1000)]

# 创建模型监控器
model_monitor = ModelMonitor(
    model_name="image_classifier",    # 模型名称
    model_version="2.0",              # 模型版本
    reference_data=reference          # 参考数据（用于漂移检测）
)

print("=" * 50)
print("模型监控器创建成功")
print("=" * 50)
print(f"模型名称: {model_monitor.model_name}")
print(f"模型版本: {model_monitor.model_version}")
print(f"参考数据: {len(reference)} 个样本")

In [ ]:
# ============================================================
# 模拟推理并记录
# ============================================================
# record_inference() 记录每次推理的输入、输出、标签和延迟
# 这些数据用于计算准确率和检测漂移

print("=" * 50)
print("模拟 500 次推理...")
print("=" * 50)

for i in range(500):
    # 模拟输入数据
    input_data = [random.random() for _ in range(10)]
    
    # 模拟预测（与参考数据相似的分布）
    prediction_score = random.gauss(0.5, 0.1)
    pred_label = 1 if prediction_score > 0.5 else 0
    
    # 模拟真实标签（80%准确率）
    label = pred_label if random.random() > 0.2 else (1 - pred_label)
    
    # 模拟延迟
    latency = random.expovariate(1/30)  # 平均30ms
    
    # 记录推理
    model_monitor.record_inference(
        input_data=input_data,    # 输入特征
        prediction=pred_label,    # 模型预测
        label=label,              # 真实标签（如果有）
        latency_ms=latency        # 推理延迟
    )

print("推理模拟完成!")
print(f"  总推理次数: 500")
print(f"  预期准确率: ~80%")

In [ ]:
# ============================================================
# 获取综合统计
# ============================================================
# get_stats() 返回模型的综合统计信息
# 包括模型元数据、请求统计、延迟统计、准确率等

stats = model_monitor.get_stats()

print("=" * 50)
print("模型监控综合统计")
print("=" * 50)
print(f"模型: {stats['model_name']} v{stats['model_version']}")
print(f"总请求: {stats['total_requests']}")

# 准确率（如果有标签数据）
if stats.get('accuracy'):
    print(f"准确率: {stats['accuracy']:.2%}")
else:
    print("准确率: N/A (无标签数据)")

print("\n延迟统计:")
print(f"  平均: {stats.get('latency_avg', 0):.2f}ms")
print(f"  P99:  {stats.get('latency_p99', 0):.2f}ms")

In [ ]:
# ============================================================
# 检查数据漂移
# ============================================================
# check_drift() 使用参考数据检测当前预测分布是否发生漂移
# 这是监控模型健康状态的重要指标

drift_result = model_monitor.check_drift(method="psi", threshold=0.2)

print("=" * 50)
print("漂移检测结果")
print("=" * 50)
print(f"检测方法: {drift_result['method']}")
print(f"漂移检测: {'是 ⚠️' if drift_result['drift_detected'] else '否 ✓'}")
if 'psi' in drift_result:
    print(f"PSI 分数: {drift_result['psi']:.4f}")
    
print("\n说明: 如果检测到漂移，可能需要重新训练模型")

In [ ]:
# ============================================================
# 获取预测分布统计
# ============================================================
# get_prediction_distribution() 返回预测值的分布统计
# 用于监控模型输出是否正常

dist = model_monitor.get_prediction_distribution()

if dist:
    print("=" * 50)
    print("预测分布统计")
    print("=" * 50)
    print(f"样本数:   {dist['count']}")
    print(f"均值:     {dist['mean']:.4f}")
    print(f"标准差:   {dist['std']:.4f}")
    print(f"最小值:   {dist['min']:.4f}")
    print(f"最大值:   {dist['max']:.4f}")
    print("\n说明: 监控预测分布可以发现模型行为异常")
else:
    print("暂无预测数据")

In [ ]:
# ============================================================
# 导出指标
# ============================================================
# export_metrics() 将指标导出为 JSON 或 Prometheus 格式
# JSON 格式便于存储和分析，Prometheus 格式用于监控系统集成

json_output = model_monitor.export_metrics(format="json")

print("=" * 50)
print("导出的 JSON 指标")
print("=" * 50)
print(json_output[:500] + "..." if len(json_output) > 500 else json_output)
print("\n说明: JSON 格式便于存储到数据库或发送到监控系统")

## 6. 工厂函数 (create_monitor)

**便捷功能**: 使用工厂函数快速创建监控器

```python
# 创建模型监控器
monitor = create_monitor(
    model_name="模型名称",
    model_version="版本号",
    reference_data=[...]  # 可选：参考数据
)
```

In [ ]:
# ============================================================
# 使用工厂函数创建监控器
# ============================================================
# create_monitor() 是创建监控器的便捷方法

monitor2 = create_monitor(
    model_name="text_classifier",
    model_version="1.0",
    reference_data=[random.random() for _ in range(100)]
)

# 记录一些数据
for _ in range(100):
    monitor2.record_inference(
        input_data=[1.0],
        prediction=random.random(),
        latency_ms=random.uniform(10, 50)
    )

print("=" * 50)
print("工厂函数创建的监控器")
print("=" * 50)
print(f"模型: {monitor2.model_name} v{monitor2.model_version}")
print(f"请求数: {monitor2.get_stats()['total_requests']}")

## 7. Prometheus 集成 (可选)

**企业级方案**: Prometheus 是业界标准的监控系统

```
Prometheus 监控架构:
┌─────────────────────────────────────────────────────────────┐
│  应用程序                                                    │
│  └── PrometheusExporter (暴露 /metrics 端点)                │
│           ↓                                                  │
│  Prometheus Server (定期拉取指标)                           │
│           ↓                                                  │
│  Grafana (可视化仪表盘)                                     │
│           ↓                                                  │
│  AlertManager (告警通知)                                    │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Prometheus 集成示例
# ============================================================
# 如果安装了 prometheus-client，可以将指标暴露给 Prometheus

if PROMETHEUS_AVAILABLE:
    from monitoring import PrometheusExporter
    print("=" * 50)
    print("Prometheus 导出器使用示例")
    print("=" * 50)
    print("""
# 1. 创建导出器
exporter = PrometheusExporter(
    port=8000,           # HTTP 端口
    prefix="ml_model"    # 指标前缀
)

# 2. 启动 HTTP 服务器
exporter.start()

# 3. 记录指标
exporter.record_prediction(
    model_name="my_model",
    model_version="1.0",
    latency_seconds=0.05
)
exporter.set_accuracy("my_model", 0.95)
exporter.set_drift_score("my_model", 0.15, method="psi")

# 4. 访问指标
# http://localhost:8000/metrics
""")
else:
    print("=" * 50)
    print("Prometheus 未安装")
    print("=" * 50)
    print("安装命令: pip install prometheus-client")
    print("\n安装后可以:")
    print("  - 暴露 /metrics 端点")
    print("  - 与 Prometheus Server 集成")
    print("  - 使用 Grafana 可视化")
    print("  - 配置 AlertManager 告警")

## 总结

本教程介绍了生产监控的核心功能：

```
核心概念回顾:
┌─────────────────────────────────────────────────────────────┐
│  DriftDetector   → 漂移检测（KS、PSI、均值/方差）           │
│  MetricsMonitor  → 指标监控（延迟、错误率）                 │
│  ModelMonitor    → 综合监控（指标+漂移+元数据）             │
│                                                             │
│  核心操作:                                                   │
│  ├── detect_drift()       检测数据漂移                      │
│  ├── record_latency()     记录延迟                          │
│  ├── record_error()       记录错误                          │
│  ├── add_alert_rule()     添加告警规则                      │
│  ├── check_alerts()       检查告警                          │
│  └── export_metrics()     导出指标                          │
└─────────────────────────────────────────────────────────────┘
```

### 最佳实践

| 实践 | 说明 |
|:-----|:-----|
| 合理阈值 | P99延迟 < 100ms，错误率 < 1% |
| 定期检测 | 每天/每周检查数据漂移 |
| 分级告警 | INFO → WARNING → ERROR → CRITICAL |
| 保留历史 | 保存指标用于趋势分析 |
| 自动化 | 告警触发自动通知和处理 |

### 常用阈值参考

| 指标 | 告警阈值 | 说明 |
|:-----|:---------|:-----|
| P99延迟 | > 100ms | 根据业务调整 |
| 错误率 | > 1% | 严重问题 |
| PSI | > 0.2 | 显著漂移 |
| 准确率下降 | > 5% | 需要重训练 |

### 下一步

- 学习 [04_DriftDetection_tutorial](04_DriftDetection_tutorial.ipynb) 深入了解漂移检测
- 学习 [05_ABTesting_tutorial](05_ABTesting_tutorial.ipynb) 了解 A/B 测试